# Bollywood Embeddings Workshop

**Can a machine understand what a movie is *about* — without being told?**

In this workshop we'll:
1. Load 2,000 Bollywood movie plots
2. Convert each plot to a **384-number vector** (an embedding)
3. Compress those vectors to 2D so we can *see* them
4. Let the machine automatically discover clusters — no genre labels used
5. Ask a question in Hindi and find matching movies

> **Runtime**: ~8 minutes on Colab free tier (CPU). Go to **Runtime → Change runtime type → T4 GPU** for ~3 minutes.
>
> Run all cells in order: **Runtime → Run all**

## Cell 1 — Install libraries

We need four packages:
- `sentence-transformers` — loads the multilingual embedding model
- `datasets` — loads the Bollywood dataset from Hugging Face in one line
- `umap-learn` — compresses 384D embeddings to 2D
- `plotly` — interactive charts (hover to see movie details)

<details>
<summary>📖 <strong>Teaching moment: what is each library actually doing?</strong></summary>

**`sentence-transformers`** is a wrapper around Hugging Face's `transformers` library. It makes it easy to load models specifically designed to produce *sentence-level* embeddings — as opposed to word-level embeddings (like Word2Vec) or token-level outputs (like raw BERT). Under the hood it runs a transformer neural network and mean-pools the token outputs into one vector per sentence.

**`datasets`** connects to the Hugging Face Hub — a public repository of thousands of ML datasets. With one call (`load_dataset(...)`) it streams or downloads the data, caches it locally, and hands you a Python object that behaves like a list of dictionaries. No manual CSV download needed.

**`umap-learn`** implements UMAP (Uniform Manifold Approximation and Projection). It's a dimensionality reduction algorithm — similar in goal to PCA, but better at preserving the *local structure* of data (i.e. which points are neighbours of which). We use it to squash 384 dimensions down to 2 so we can draw a picture.

**`plotly`** produces interactive HTML charts. Unlike `matplotlib`, Plotly charts let users zoom, pan, and hover over individual dots to see the data behind them — essential for exploring a 2,000-point scatter plot.

</details>

In [ ]:
!pip install sentence-transformers datasets umap-learn plotly pandas -q

## Cell 2 — Load the Bollywood dataset

The dataset has 4,061 Bollywood movies with `title`, `plot`, and `genres`.
We sample 2,000 to keep things fast.

**Before we cluster**: look at the genre distribution — what categories exist?

<details>
<summary>📖 <strong>Teaching moment: random sampling and reproducibility</strong></summary>

**Why sample at all?**
Embedding 4,061 plots on Colab's free CPU would take ~15 minutes. 2,000 is enough to see clear structure while staying fast. In production systems, researchers often prototype on a sample, verify the approach works, then scale up.

**What does `random_state=42` do?**
Random sampling picks different rows every time — unless you fix a *seed*. Setting `random_state=42` (any fixed number works) makes the sample identical for everyone who runs the notebook. This is essential in teaching and research: it means every student sees the same clusters, making discussion easier.

**What is the Hugging Face Hub?**
Think of it like GitHub, but for ML models and datasets. Anyone can upload a dataset; anyone can download it with `load_dataset("username/dataset-name")`. The Hub hosts hundreds of thousands of datasets in dozens of languages. `Vangmayy/bollywood_plots` was uploaded by a community contributor — one person's effort, usable by everyone.

**Why look at genre distribution first?**
Before doing any ML, it's good practice to *look at your data*. If 90% of movies are "Drama", the model has very little variety to discover. Knowing the distribution helps you interpret the clusters you'll see later.

</details>

In [ ]:
from datasets import load_dataset
import pandas as pd
import ast

ds = load_dataset("Vangmayy/bollywood_plots", split="train")
df = ds.to_pandas().sample(2000, random_state=42).reset_index(drop=True)

# --- Parse the raw columns ---
# 'input'  → genres  (stored as a stringified Python list, e.g. "['Action', 'Drama']")
# 'output' → title + plot  (stored as "title: Foo\nStory: Bar baz...")

def parse_genres(raw):
    try:
        return ast.literal_eval(raw)
    except Exception:
        return []

def parse_title(raw):
    # First line: "title: <name>"
    line = raw.split("\n")[0]
    return line.replace("title:", "").strip()

def parse_plot(raw):
    # Everything after the first "\nStory: "
    parts = raw.split("\nStory:", 1)
    return parts[1].strip() if len(parts) > 1 else ""

df["genres"] = df["input"].apply(parse_genres)
df["title"]  = df["output"].apply(parse_title)
df["plot"]   = df["output"].apply(parse_plot)

# Drop the raw columns we no longer need
df = df.drop(columns=["Unnamed: 0", "input", "output"])

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

In [ ]:
# What genres exist in the data?
from collections import Counter

all_genres = [g for genres in df["genres"] for g in (genres if genres else [])]
genre_counts = Counter(all_genres).most_common(15)

print("Top 15 genres in the dataset:")
for genre, count in genre_counts:
    bar = "█" * (count // 10)
    print(f"  {genre:<20} {count:>4}  {bar}")

## Cell 3 — Embed the movie plots

We use **`paraphrase-multilingual-MiniLM-L12-v2`** — a model trained on 50+ languages including Hindi.
It converts any text to a vector of **384 numbers**.

Think of each number as measuring one abstract "dimension of meaning".
Two plots about detectives will have similar numbers; a detective plot and a romance plot will differ.

> **Why multilingual?** So Cell 8 works — you can type a Hindi query and it lands in the same space as the English plots.

<details>
<summary>📖 <strong>Teaching moment: what is an embedding, really?</strong></summary>

**The core idea**
An embedding is a function that maps *anything* (a word, a sentence, an image, a user's watch history) to a fixed-length list of numbers. The numbers aren't random — they're chosen so that *similar things produce similar numbers*. That's it. That one property is what makes embeddings powerful.

**Where do those 384 numbers come from?**
The model is a transformer neural network with ~12 layers. Each layer transforms the input a little more. The final output is a 384-dimensional vector. The model was trained (by someone else, not us) on millions of sentence pairs labelled as "similar" or "different". Through training, it learned to compress meaning into 384 numbers in a way that preserves similarity. We just use it — like a calculator.

**Why 384 dimensions?**
It's a design choice by the model's creators. Larger models use 768 or 1536 dimensions and capture more nuance; smaller models use fewer and run faster. 384 is a sweet spot for speed vs. quality. The exact value doesn't matter much for understanding the concept.

**The multilingual trick**
`paraphrase-multilingual-MiniLM-L12-v2` was trained on text in 50+ languages *mapped to the same embedding space*. This means "detective" in English and "जासूस" in Hindi end up as nearby vectors. So when you embed a Hindi query in Cell 8, it lands near English movie plots with similar meaning — cross-language similarity for free.

**What does "inference" mean?**
We are *not* training the model — we're running it on new inputs. Training would mean adjusting the model's weights, which requires days on expensive hardware. Inference (what we do here) just runs a forward pass through the fixed network — fast even on a laptop CPU.

</details>

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Embedding 2,000 movie plots...")
embeddings = model.encode(
    df["plot"].tolist(),
    show_progress_bar=True,
    batch_size=64
)

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"→ {embeddings.shape[0]} movies, each represented by {embeddings.shape[1]} numbers")

## Cell 4 — Compress to 2D with UMAP

We can't plot 384 dimensions. UMAP squashes them to 2 while trying to keep "nearby" points nearby.

**Analogy**: imagine flattening a 3D globe onto a 2D map. Some distortion is unavoidable,
but countries that were neighbours stay neighbours.

UMAP does the same for meaning-space.

<details>
<summary>📖 <strong>Teaching moment: UMAP vs PCA — why does it matter?</strong></summary>

**Why not just use PCA?**
PCA (Principal Component Analysis) is the classical method for dimensionality reduction. It finds the directions of maximum *global* variance and projects data onto them. It's fast and interpretable, but it's a *linear* method — it can only find straight-line structure. If the data curves or folds in high-dimensional space, PCA will distort it badly.

UMAP is non-linear. It builds a graph of nearest neighbours in high-dimensional space, then finds a 2D layout that preserves those neighbourhood relationships as faithfully as possible. The result looks much more like the real structure of the data.

**The parameters we set:**
- `n_neighbors=15` — how many nearest neighbours to consider when building the graph. Higher = more global structure preserved; lower = more fine-grained local clusters.
- `min_dist=0.1` — how tightly points can be packed together in 2D. Lower values create tighter, more separated clusters.
- `metric="cosine"` — how we measure distance between embeddings. Cosine similarity measures the *angle* between vectors, which is better than Euclidean distance for high-dimensional text embeddings (where all vectors are roughly the same length).

**Why does UMAP take longer than the embedding step?**
Embedding is a single forward pass through a neural network — highly parallelisable on GPU or CPU. UMAP builds a nearest-neighbour graph over all 2,000 points, which requires comparing every point to many others — it scales roughly as O(N × log N). For 2,000 points it takes ~1 minute; for 50,000 points it would take ~10 minutes.

**Important caveat**
The 2D positions are a *projection* — exact distances in 2D don't perfectly reflect exact distances in 384D. Don't read too much into the precise coordinates. What matters is the overall cluster structure and which movies are in the same neighbourhood.

</details>

In [ ]:
import umap

print("Running UMAP (this takes ~1-2 minutes on CPU)...")
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)
coords = reducer.fit_transform(embeddings)

df["x"] = coords[:, 0]
df["y"] = coords[:, 1]

print(f"Done. Each movie now has an (x, y) position in 2D space.")
df[["title", "x", "y"]].head()

## Cell 5 — Auto-cluster with KMeans

KMeans divides the 2,000 movies into 8 groups based purely on their 384D embeddings.

**We never mentioned genres.** The algorithm only sees numbers — yet it groups movies that are semantically similar. Let's see if those groups match real genres.

<details>
<summary>📖 <strong>Teaching moment: how does KMeans work, and what is unsupervised learning?</strong></summary>

**KMeans in plain English**
1. Place 8 "centre" points randomly in 384D space.
2. Assign every movie to its nearest centre.
3. Move each centre to the average position of the movies assigned to it.
4. Repeat steps 2–3 until the centres stop moving.

The result: 8 groups where movies within a group are close to each other in embedding space, and groups are as far apart from each other as possible.

**Why 8 clusters?**
We chose 8 as a starting point — roughly matching the number of major Bollywood genres. In real applications, you'd use methods like the *elbow method* or *silhouette score* to find the best K. Try changing 8 to 5 or 12 in Cell 5 and re-running — the clusters will reorganise.

**Supervised vs. unsupervised learning**
- **Supervised**: you give the algorithm labelled examples ("this is Action", "this is Romance") and it learns to predict labels for new data. Requires human annotation.
- **Unsupervised**: you give the algorithm only the data, no labels. It finds structure on its own. That's what we're doing here — KMeans has never seen the word "genre".

The fact that unsupervised clusters often align with human-defined categories (like genre) is what makes embeddings so powerful: the model has learned a representation of meaning that mirrors how humans categorise things, without being explicitly taught those categories.

**Note on `n_init=10`**
KMeans is sensitive to where the initial centres are placed. We run it 10 times with different random starts and keep the best result. This is why you'll see it's slightly slower than you might expect.

</details>

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=8, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(embeddings).astype(str)

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())

# Peek at what's in each cluster — sample one title per cluster
print("\nSample title from each cluster:")
for c in sorted(df["cluster"].unique()):
    sample = df[df["cluster"] == c].iloc[0]
    genres = sample["genres"][:2] if sample["genres"] else ["?"]
    print(f"  Cluster {c}: {sample['title']}  [{', '.join(genres)}]")

## Cell 6 — Interactive map (coloured by cluster)

Each dot is a movie. Hover over any dot to read its title, genres, and plot summary.

**Look for**: do clusters look visually tight? Do nearby dots share a genre or theme?

In [ ]:
import plotly.express as px

# Truncate long plots for cleaner hover text
df["plot_short"] = df["plot"].str[:200] + "..."
df["genres_str"] = df["genres"].apply(lambda g: ", ".join(g) if g else "Unknown")

fig = px.scatter(
    df,
    x="x", y="y",
    color="cluster",
    hover_data={"title": True, "genres_str": True, "plot_short": True, "x": False, "y": False},
    labels={"genres_str": "Genres", "plot_short": "Plot"},
    title="Bollywood Movies — Embedding Space (UMAP) · Coloured by Auto-Discovered Cluster",
    width=950, height=720
)
fig.update_traces(marker=dict(size=5, opacity=0.75))
fig.update_layout(legend_title_text="Cluster")
fig.show()

## Cell 7 — Same map, coloured by genre

Now we colour the **exact same dots** by their actual primary genre (from the dataset).

**Key question**: Do the auto-discovered clusters (Cell 6) align with the genre colours here?
If yes — the model figured out genres from meaning alone, without ever being told.

<details>
<summary>📖 <strong>Teaching moment: the "aha" moment — what alignment tells us</strong></summary>

**What you're looking for**
Compare the two charts side by side (or toggle between them):
- **Cell 6**: colours = machine-discovered clusters (0–7)
- **Cell 7**: colours = human-assigned genres (Action, Drama, Romance…)

If large regions of one colour in Cell 7 roughly correspond to single clusters in Cell 6 — the model has recovered human categories from raw text, without supervision.

**Why this is surprising**
The embedding model was never trained on this Bollywood dataset. It was trained on generic multilingual text. KMeans was never told what genres exist. Yet together, they can separate Action movies from Romance movies by reading the plots. This is because *meaning is structure* — thrillers use different vocabulary, sentence structure, and narrative patterns than romances, and the model has learned to detect those patterns.

**Why won't it be a perfect match?**
Several reasons:
1. Genre labels are human-assigned and imperfect — many movies are multi-genre.
2. Two movies can be in the same genre but have very different plots (e.g. a romantic comedy set in space vs. a small-town love story).
3. UMAP introduces some distortion in the 2D projection.
4. KMeans assumes spherical clusters — real semantic clusters are often irregular shapes.

A good partial match is actually the expected outcome, and it's still a meaningful result.

**Real-world analogy**
This is essentially how Netflix and Spotify recommendation engines work — without ever hard-coding genres. They embed user behaviour or content descriptions, find clusters of similar items, and recommend within clusters. The categories emerge from data rather than being defined in advance.

</details>

In [ ]:
df["primary_genre"] = df["genres"].apply(lambda g: g[0] if g else "Unknown")

# Keep only genres with enough movies for a readable legend
top_genres = df["primary_genre"].value_counts().nlargest(12).index
df["genre_label"] = df["primary_genre"].apply(lambda g: g if g in top_genres else "Other")

fig2 = px.scatter(
    df,
    x="x", y="y",
    color="genre_label",
    hover_data={"title": True, "genres_str": True, "plot_short": True, "x": False, "y": False},
    labels={"genre_label": "Primary Genre", "genres_str": "Genres", "plot_short": "Plot"},
    title="Same Map — Coloured by Primary Genre",
    width=950, height=720
)
fig2.update_traces(marker=dict(size=5, opacity=0.75))
fig2.show()

## Cell 8 — Try it yourself (Hindi / Hinglish / English)

Type **any description** of a movie you'd like to find — in Hindi, Hinglish, or English.
The model embeds your query into the same 384D space and finds the 5 nearest movies.

**Examples to try**:
- `"एक जासूस अपराधियों को पकड़ता है"` — A detective catches criminals
- `"romantic love story college"` 
- `"family drama emotional reunion"`
- `"action hero saves the country"`
- `"supernatural ghost haunted house"`

<details>
<summary>📖 <strong>Teaching moment: cosine similarity and nearest-neighbour search</strong></summary>

**How does "find nearest movies" work?**
When you type a query, the model converts it to a 384D vector — exactly like it did for the movie plots. We then measure the *similarity* between your query vector and every movie's vector. The 5 with the highest similarity scores are returned.

**What is cosine similarity?**
Cosine similarity measures the *angle* between two vectors, not their length:

```
cosine_similarity(A, B) = (A · B) / (|A| × |B|)
```

- Score = **1.0** → vectors point in the same direction → same meaning
- Score = **0.0** → vectors are perpendicular → unrelated meaning
- Score = **-1.0** → vectors point opposite directions → opposite meaning

We use cosine (not Euclidean distance) because text embeddings tend to cluster on the surface of a high-dimensional sphere — what matters is direction, not magnitude.

**Why does Hindi work?**
Because this model was trained on parallel data (the same sentences in multiple languages) so Hindi and English sentences with the same meaning produce similar vectors. Your Hindi query vector will be close to the English plot vectors that describe the same kind of story. This is called a *multilingual embedding space*.

**What would break it?**
- A query that describes something completely outside the training data (e.g. a very specific local cultural reference the model hasn't seen)
- A very short, ambiguous query like "love" — too many movies are about love
- A typo-heavy query in an unusual dialect

**This pattern has a name: semantic search**
Keyword search (like old Google) looks for exact word matches. Semantic search finds meaning-matches even with different words. Almost every modern search system — Google, YouTube, Spotify, Amazon — uses embedding-based semantic search under the hood.

</details>

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# ← Change this to any description you like!
query = "एक जासूस अपराधियों को पकड़ता है"

q_emb = model.encode([query])
sims = cosine_similarity(q_emb, embeddings)[0]
top_idx = sims.argsort()[-5:][::-1]

print(f"Query: {query}")
print(f"\nTop 5 nearest Bollywood movies:\n")

results = df.iloc[top_idx][["title", "genres_str", "plot_short"]].copy()
results["similarity"] = sims[top_idx].round(3)
results = results.reset_index(drop=True)
results.index += 1
display(results)

In [ ]:
# Visualise WHERE your query lands on the map
q_2d = reducer.transform(q_emb)

import plotly.graph_objects as go

fig3 = px.scatter(
    df, x="x", y="y",
    color="cluster",
    opacity=0.4,
    hover_data={"title": True, "genres_str": True, "x": False, "y": False},
    title=f"Your query lands here: \"{query}\"",
    width=950, height=720
)
fig3.update_traces(marker=dict(size=4))

# Mark the query point
fig3.add_trace(go.Scatter(
    x=[q_2d[0, 0]], y=[q_2d[0, 1]],
    mode="markers+text",
    marker=dict(size=18, color="black", symbol="star"),
    text=["← your query"],
    textposition="middle right",
    name="Query",
    showlegend=True
))

# Mark the top 5 nearest movies
near = df.iloc[top_idx]
fig3.add_trace(go.Scatter(
    x=near["x"], y=near["y"],
    mode="markers+text",
    marker=dict(size=12, color="red", symbol="circle-open", line=dict(width=2)),
    text=near["title"].str.slice(0, 20),
    textposition="top center",
    name="Top 5 matches",
    showlegend=True
))

fig3.show()

## Cell 9 — (Optional) Save your results

Save the 2D coordinates + clusters as a Parquet file.
If you have a Hugging Face account you can upload your own version of the dataset.

In [ ]:
out = df[["title", "genres_str", "plot", "x", "y", "cluster", "primary_genre"]].copy()
out.to_parquet("bollywood_2d.parquet", index=False)
print("Saved to bollywood_2d.parquet")
print(out.shape)

# To upload to Hugging Face (needs HF account + token):
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="bollywood_2d.parquet",
#     path_in_repo="bollywood_2d.parquet",
#     repo_id="your-username/bollywood-embeddings",
#     repo_type="dataset",
# )

## Cell 10 — Interactive map with Embedding Atlas

Embedding Atlas is a dedicated tool for visually exploring large text collections.
Because we already computed `x` and `y` in Cell 4, we pass those directly — no re-embedding.

What you get over the Plotly chart:
- Pan, zoom, and lasso-select clusters
- Full-text search across all 2,000 movies
- Sidebar with column statistics
- Colour by any column (cluster, genre, etc.)

In [ ]:
!pip install embedding-atlas -q

from embedding_atlas.widget import EmbeddingAtlasWidget

# df already has x, y from Cell 4 — no re-embedding needed
widget = EmbeddingAtlasWidget(
    df[["title", "genres_str", "plot_short", "cluster", "primary_genre", "x", "y"]],
    x="x",
    y="y",
)
widget

## Discussion — Where does this show up in real life?

| Application | What embeddings do |
|-------------|-------------------|
| **Netflix / Hotstar recommendations** | Embed movie descriptions; suggest nearest neighbours |
| **Google Search** | Embed your query + web pages; return nearest pages |
| **WhatsApp spam filter** | Embed message; check if it's near known spam |
| **ChatGPT memory** | Embed past conversations; retrieve relevant ones |
| **Duolingo** | Embed answers; check if meaning matches (not just exact words) |

Every time a system needs to understand *meaning* — not just match keywords — it probably uses embeddings.

---

### Things to explore
- Change `n_clusters` in Cell 5 from 8 to 12 or 5. How do the clusters change?
- Try a completely wrong query (e.g. a cricket match description). Where does it land?
- Find a movie you know well. Does it cluster with movies you'd expect?
- Change the model to `all-MiniLM-L6-v2` (English-only). Does the Hindi query still work?